# Coding practice: trace and diagnose RNN gradients

A linear recurrent step is `h_t = w * h_{t-1} + x_t`. Backpropagating through T steps multiplies the recurrent weight `w` T times. Implement that product, verify several regimes, and locate a vanishing threshold.

The course-page check uses a changed `(w, T)` pair and a mitigation decision, so reason from the trace rather than copy one printed number.

> Save your own copy first.

In [ ]:
def gradient_factor(w, steps):
    """Return d h_T / d h_0 = the product of the recurrent weight over `steps` steps."""
    factor = 1.0
    for t in range(steps):
        # TODO: each step back multiplies the running factor by w
        raise NotImplementedError("Multiply the running factor by w each step.")
    return factor

In [ ]:
cases = [(0.5, 10), (0.8, 6), (1.0, 12), (1.2, 10)]
observed = {(w, steps): gradient_factor(w, steps) for w, steps in cases}
for (w, steps), value in observed.items():
    assert abs(value - w**steps) < 1e-12, f"Mismatch for w={w}, steps={steps}"
assert observed[(0.5, 10)] < 0.001
assert observed[(1.0, 12)] == 1.0
assert observed[(1.2, 10)] > 1.0
for key, value in observed.items():
    print(f"w={key[0]:.1f}, T={key[1]:2d} -> {value:.6f}")
print("All bounded gradient-factor checks passed.")

In [ ]:
def first_step_below(w, threshold, max_steps=200):
    for steps in range(1, max_steps + 1):
        if abs(gradient_factor(w, steps)) < threshold:
            return steps
    return None

threshold = 0.01
first_vanishing_step = first_step_below(0.8, threshold)
assert first_vanishing_step == 21
print(f"For w=0.8, |w|^T first falls below {threshold} at T={first_vanishing_step}.")

## Interpret the trace

Use the threshold and table as evidence:

1. Predict whether `w=0.9` crosses `0.01` earlier or later than `w=0.8`, then test it.
2. Gradient clipping limits an optimizer update after an explosion. Why does it not change the repeated Jacobian that caused the trace?
3. Name one intervention that changes the long path itself (for example, a gated/additive state path or an orthogonal/spectrally controlled transition) and one diagnostic you would rerun.

The follow-up asks you to separate immediate protection from evidence about the underlying recurrent dynamics.